<a href="https://colab.research.google.com/github/ThinkingBeyond/BeyondAI-2025/blob/main/Tornike%20Khabeishvili%20and%20Mohamed%20Hassan/OU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

==============================================================================
ORNSTEIN-UHLENBECK (OU) PROCESS BENCHMARK
==============================================================================
PURPOSE: This script benchmarks different solvers from the Diffrax library by
training a Neural SDE model on the Ornstein-Uhlenbeck (OU) process. The OU
process is a mean-reverting stochastic process commonly used in finance and
physics, making it an ideal test case for SDE solvers.

WHAT IT DOES:
- Generates synthetic training data from the OU process SDE system
- Trains a Neural SDE model to learn the dynamics using multiple solvers
- Tracks and compares: training time, compilation time, memory usage (CPU/GPU)
- Creates comprehensive visualization plots for performance comparison

KEY FEATURES:
- Accurate time measurement (excludes JIT compilation from training time)
- True peak GPU memory tracking via background monitoring at 100 Hz
- CPU memory tracking via tracemalloc

ORNSTEIN-UHLENBECK PROCESS:
A classic mean-reverting stochastic process:
  dX = θ(μ - X) dt + σ dW

where:
  - θ (theta): Mean reversion speed (how fast X returns to μ)
  - μ (mu): Long-term mean (equilibrium level)
  - σ (sigma): Volatility (noise intensity)
  - W: Wiener process (Brownian motion)

When X > μ, the drift pulls X downward toward μ
When X < μ, the drift pulls X upward toward μ

ARCHITECTURE:
- Neural SDE with drift (deterministic) and diffusion (stochastic) terms
- Uses Equinox for neural network layers
- Uses Diffrax for SDE solving

In [ ]:
!pip install jax==0.7.2 jaxlib==0.7.2
!pip install equinox==0.13.2
!pip install git+https://github.com/daniil-shmelev/diffrax
!pip install optax==0.2.6
!pip install lineax==0.0.8
!pip install numpy==2.0.2
!pip install matplotlib==3.10.0
!pip install pynvml==13.0.1

In [ ]:
import os
import random
import pickle
import time
import tracemalloc
import threading
from contextlib import contextmanager

# ============================================================================
# JAX CONFIGURATION - Prevent Memory Pre-allocation
# ============================================================================
# This disables JAX's memory pre-allocation
# Has to be done, because we want to compare the actual usage of memory by each solver
# If this is not done, all solvers will use the same amount of GPU memory
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['XLA_PYTHON_CLIENT_ALLOCATOR'] = 'platform'

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
import jax.random as jr
import jax.nn as jnn
import equinox as eqx           # Neural network library for JAX
import diffrax                  # Differential equation solver
import optax
import lineax as lx             # Linear algebra operations for JAX

In [ ]:
# Try to import GPU memory tracking
# Had issues on my machine with memory tracking in the beginning
try:
    try:
        import pynvml
    except ImportError:
        import nvidia_smi as pynvml

    pynvml.nvmlInit()
    GPU_AVAILABLE = True
    GPU_HANDLE = pynvml.nvmlDeviceGetHandleByIndex(0)
    # Test that it works
    test_info = pynvml.nvmlDeviceGetMemoryInfo(GPU_HANDLE)
    print(f"✓ GPU memory tracking enabled: {test_info.used / (1024**3):.2f} GB currently used")
except ImportError:
    GPU_AVAILABLE = False
    GPU_HANDLE = None
    print("=" * 60)
    print("⚠️  WARNING: GPU tracking library not installed!")
    print("GPU memory tracking is DISABLED")
    print("Install with: pip install pynvml")
    print("=" * 60)
except Exception as e:
    GPU_AVAILABLE = False
    GPU_HANDLE = None
    print("=" * 60)
    print(f"⚠️  WARNING: Could not initialize GPU tracking: {e}")
    print("GPU memory tracking is DISABLED")
    print("=" * 60)

If you get a "Could not initialize GPU tracking: NVML Shared Library Not Found
GPU memory tracking is DISABLED" error on Colab, change runtime type to T4 GPU

In [ ]:
def get_gpu_memory_mb():
    """Get current GPU memory usage in MB."""
    if not GPU_AVAILABLE:
        return 0.0
    try:
        info = pynvml.nvmlDeviceGetMemoryInfo(GPU_HANDLE)
        return info.used / (1024 * 1024)
    except:
        return 0.0


class BackgroundMemoryMonitor:
    """
    Continuously monitor GPU memory in background thread to catch true peaks.
    Samples at high frequency (100 Hz) to capture spikes during computation.
    """
    def __init__(self, interval=0.01):
        self.peak = 0.0
        self.start_memory = 0.0
        self.running = False
        self.interval = interval
        self.history = []
        self.step_count = 0

    def _monitor_loop(self):
        """Background thread that continuously samples GPU memory."""
        while self.running:
            current = get_gpu_memory_mb()
            self.peak = max(self.peak, current)
            time.sleep(self.interval)

    def start_monitoring(self):
        """Start background monitoring thread."""
        self.start_memory = get_gpu_memory_mb()
        self.peak = self.start_memory
        self.running = True
        self.history = [(0, self.start_memory)]
        self.step_count = 0
        self.thread = threading.Thread(target=self._monitor_loop, daemon=True)
        self.thread.start()

    def record_step(self):
        """Record current memory for time-series (called after each batch)."""
        current = get_gpu_memory_mb()
        self.step_count += 1
        self.history.append((self.step_count, current))

    def stop_monitoring(self):
        """Stop background monitoring and return results."""
        self.running = False
        time.sleep(self.interval * 2)  # Let final samples complete
        return self.peak

    def get_peak_delta(self):
        """Get peak memory above baseline."""
        return self.peak - self.start_memory

    def get_history(self):
        """Return memory history as (steps, memory_values)."""
        if not self.history:
            return [], []
        steps, memory = zip(*self.history)
        return list(steps), list(memory)


def format_memory(bytes_val):
    """Format memory in human-readable form."""
    mb = bytes_val / (1024 * 1024)
    if mb < 1024:
        return f"{mb:.2f} MB"
    else:
        gb = mb / 1024
        return f"{gb:.2f} GB"

In [ ]:
# ============================================================================
# ORIGINAL CODE
# ============================================================================

def seed_everything(seed):
    """Set random seeds for reproducibility."""
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    random.seed(seed)


def ou_drift(t, y, args):
    """Drift term for OU process: theta * (mu - X)"""
    theta, mu, sigma = args
    return theta * (mu - y)


def ou_diffusion(t, y, args):
    """Diffusion term for OU process: sigma"""
    theta, mu, sigma = args
    return lx.DiagonalLinearOperator(sigma * jnp.ones_like(y))


def generate_ou_path(key, T, N, theta, mu, sigma, X0):
    """Generate a single OU process path using Diffrax."""
    t = jnp.linspace(0, T, N)
    dt = T / (N - 1)
    y0 = jnp.array([X0])
    bm = diffrax.VirtualBrownianTree(t0=0, t1=T, tol=dt/2, shape=(1,), key=key)
    drift = diffrax.ODETerm(lambda t, y, args: ou_drift(t, y, args))
    diffusion = diffrax.ControlTerm(
        lambda t, y, args: ou_diffusion(t, y, args),
        bm
    )
    terms = diffrax.MultiTerm(drift, diffusion)
    solver = diffrax.Euler()
    saveat = diffrax.SaveAt(ts=t)
    sol = diffrax.diffeqsolve(
        terms,
        solver,
        t0=0,
        t1=T,
        dt0=dt,
        y0=y0,
        args=(theta, mu, sigma),
        saveat=saveat
    )
    return t, sol.ys.squeeze()


def generate_data(config, key):
    """Generate multiple OU process samples using vectorization."""
    keys = jr.split(key, config['num_samples'])

    def generate_single(key):
        t, X = generate_ou_path(
            key,
            config['T'],
            config['N'],
            config['theta'],
            config['mu'],
            config['sigma'],
            config['X0']
        )
        return jnp.stack([t, X], axis=1)

    print("Compiling vectorized OU generation...")
    data_array = jax.vmap(generate_single)(keys)
    print("Generation complete!")

    times = jnp.linspace(0, config['T'], config['N'])

    return data_array, times


def split_data(data, train_ratio=0.8, key=None):
    """Split data into train and test sets."""
    total_size = len(data)
    train_size = int(total_size * train_ratio)

    if key is None:
        key = jr.PRNGKey(0)

    indices = jr.permutation(key, total_size)
    train_idx = indices[:train_size]
    test_idx = indices[train_size:]

    train_data = data[train_idx]
    test_data = data[test_idx]

    return train_data, test_data


def create_data_loader(data, batch_size, shuffle=True, key=None):
    """Create a simple data loader for JAX."""
    num_samples = len(data)

    if shuffle:
        if key is None:
            key = jr.PRNGKey(0)
        indices = jr.permutation(key, num_samples)
        data = data[indices]

    num_batches = num_samples // batch_size

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = start_idx + batch_size
        yield data[start_idx:end_idx]

    if num_samples % batch_size != 0:
        yield data[num_batches * batch_size:]

In [ ]:
# ============================================================================
# NEURAL SDE ARCHITECTURE
# ============================================================================
def lipswish(x):
    """
    Lipschitz-continuous activation function.

    A scaled version of SiLU (Swish) that has a Lipschitz constant ≤ 1.
    This helps with training stability
    """
    return 0.909 * jnn.silu(x)


class VectorField(eqx.Module):
    """
    The drift function f(t, y, x_t) for the Neural SDE.

    This network learns the deterministic part of the dynamics:
      dy/dt = f(t, y, x_t) + g(t) dW

    The network takes as input:
      - Current hidden state y
      - Observed data at time t (x_t from control path)

    Attributes:
        scale: Optional learnable scaling factor
        mlp: Multi-layer perceptron for computing drift
        data_size: Dimension of observed data
    """
    scale: jnp.ndarray | int
    mlp: eqx.nn.MLP
    data_size: int

    def __init__(self, data_size, hidden_size, width_size, depth, scale, *, key, **kwargs):
        super().__init__(**kwargs)
        scale_key, mlp_key = jr.split(key)
        if scale:
            self.scale = jr.uniform(scale_key, (hidden_size,), minval=0.9, maxval=1.1)
        else:
            self.scale = 1

        self.data_size = data_size

        self.mlp = eqx.nn.MLP(
            in_size=hidden_size + self.data_size,
            out_size=hidden_size,
            width_size=width_size,
            depth=depth,
            activation=lipswish,
            final_activation=jnn.tanh,
            key=mlp_key,
        )

    def __call__(self, t, y, args):
        control_path = args
        x_t = control_path.evaluate(t)
        mlp_input = jnp.concatenate([y, x_t])
        return self.scale * self.mlp(mlp_input)


class ControlledVectorField(eqx.Module):
    """
    The diffusion function g(t) for the Neural SDE.

    This network learns the stochastic part of the dynamics:
      dy/dt = f(t, y, x_t) + g(t) dW

    g(t) is a diagonal matrix (different noise intensity per dimension).

    Attributes:
        scale: Optional learnable scaling factor
        mlp: Multi-layer perceptron for computing diffusion
        hidden_size: Dimension of hidden state
    """
    scale: jnp.ndarray | int
    mlp: eqx.nn.MLP
    hidden_size: int

    def __init__(self, hidden_size, width_size, depth, scale, *, key, **kwargs):
        super().__init__(**kwargs)
        scale_key, mlp_key = jr.split(key)
        if scale:
            self.scale = jr.uniform(scale_key, (hidden_size,), minval=0.9, maxval=1.1)
        else:
            self.scale = 1

        self.mlp = eqx.nn.MLP(
            in_size=1,
            out_size=hidden_size,
            width_size=width_size,
            depth=depth,
            activation=lipswish,
            final_activation=jnn.tanh,
            key=mlp_key,
        )
        self.hidden_size = hidden_size

    def __call__(self, t, y, args):
        t_scalar = jnp.asarray(t)
        diagonal = self.mlp(t_scalar[None])
        return lx.DiagonalLinearOperator(self.scale * diagonal)


class NeuralSDE(eqx.Module):
    """
    Complete Neural SDE model for learning stochastic dynamics.

    ARCHITECTURE:
    1. initial: Encode first observation into hidden state
    2. vf: Drift function (deterministic dynamics)
    3. cvf: Diffusion function (stochastic dynamics)
    4. readout: Decode hidden state back to observation space

    The model learns to predict future observations by solving:
      dy = f(t, y, x_t) dt + g(t) dW
    where y is a learned hidden representation.

    Attributes:
        initial: MLP for initial encoding
        vf: Vector field (drift)
        cvf: Controlled vector field (diffusion)
        readout: Linear layer for decoding
        noise_size: Dimension of Brownian motion
        data_size: Dimension of observations
    """
    initial: eqx.nn.MLP
    vf: VectorField
    cvf: ControlledVectorField
    readout: eqx.nn.Linear
    noise_size: int
    data_size: int

    def __init__(self, data_size, noise_size, hidden_size, width_size, depth, *, key, **kwargs):
        super().__init__(**kwargs)
        initial_key, vf_key, cvf_key, readout_key = jr.split(key, 4)

        self.data_size = data_size

        # Encoder: observation -> hidden state
        self.initial = eqx.nn.MLP(
            data_size, hidden_size, width_size, depth, key=initial_key
        )

        # Drift
        self.vf = VectorField(data_size, hidden_size, width_size, depth, scale=True, key=vf_key)

        # Diffusion
        self.cvf = ControlledVectorField(hidden_size, width_size, depth, scale=True, key=cvf_key)

        # Decoder: hidden state -> observation
        self.readout = eqx.nn.Linear(hidden_size, data_size, key=readout_key)
        self.noise_size = noise_size

    def __call__(self, ts, control_path, solver, *, key):
        t0 = ts[0]
        t1 = ts[-1]
        dt0 = 0.05              # IMPORTANT: 0.05 works fine with avaialable computational resources
        bm_key = key            # Can be increased to 0.005 or 0.001, but network will learn very fast
                                # and consume a lot of memory/take a lot of time

        x0 = control_path.evaluate(t0)
        y0 = self.initial(x0)

        control = diffrax.VirtualBrownianTree(
            t0=t0, t1=t1, tol=dt0 / 2, shape=(self.noise_size,), key=bm_key
        )


        # Define SDE terms
        vf = diffrax.ODETerm(self.vf)
        cvf = diffrax.ControlTerm(self.cvf, control)
        terms = diffrax.MultiTerm(vf, cvf)
        saveat = diffrax.SaveAt(ts=ts)

        # Solve the Neural SDE in hidden space
        # diffeqsolve is a key part of this
        # for a good understanding, please read diffrax documentation
        # https://docs.kidger.site/diffrax/api/diffeqsolve/
        # In general, for a better understanding of the project, we recommend
        # taking a look at the diffrax library docs, especially the term structure
        sol = diffrax.diffeqsolve(
            terms, solver, t0, t1, dt0, y0,
            args=control_path,
            saveat=saveat
        )

        return jax.vmap(self.readout)(sol.ys)

In [ ]:
# ============================================================================
# TRAINING FUNCTIONS
# ============================================================================
def mse_loss(pred, true):
    """
    Mean squared error loss function.

    Args:
        pred: Predicted values
        true: True values

    Returns:
        Scalar loss value
    """
    return jnp.mean((pred - true) ** 2)

@eqx.filter_jit
def batch_loss(model, batch_data, times, key, solver):
    """
    Compute loss for a batch of trajectories.

    For each trajectory in the batch:
    1. Create interpolation of observed data (control path)
    2. Run Neural SDE forward to get predictions
    3. Compute loss between predictions and true observations
    4. Average over batch

    Args:
        model: NeuralSDE model
        batch_data: Batch of shape
        times: Time points for prediction
        key: JAX random key
        solver: Diffrax solver

    Returns:
        Scalar loss averaged over batch
    """
    keys = jr.split(key, len(batch_data))

    @jax.jit
    def make_interp(data_row):
        t = data_row[:, 0]
        x = data_row[:, 0:2]
        coeffs = diffrax.backward_hermite_coefficients(t, x)
        return diffrax.CubicInterpolation(t, coeffs)

    control_paths = jax.vmap(make_interp)(batch_data)

    def single_loss(data, control_path, key):
        true = data[:, 1]
        pred = model(times, control_path, solver, key=key)[:, 1]
        return mse_loss(pred, true)

    losses = jax.vmap(single_loss)(batch_data, control_paths, keys)
    return jnp.mean(losses)


@eqx.filter_jit
def make_step(model, opt_state, optimizer, batch_data, times, key, solver):
    """Single training step."""
    loss, grads = eqx.filter_value_and_grad(batch_loss)(model, batch_data, times, key, solver)
    updates, opt_state = optimizer.update(grads, opt_state, model)
    model = eqx.apply_updates(model, updates)
    return model, opt_state, loss


def evaluate_model(model, test_data, times, key, solver, batch_size=100):
    """Evaluate model on test data."""
    keys = jr.split(key, len(test_data))

    @jax.jit
    def make_interp(data_row):
        t = data_row[:, 0]
        x = data_row[:, 0:2]
        coeffs = diffrax.backward_hermite_coefficients(t, x)
        return diffrax.CubicInterpolation(t, coeffs)

    all_control_paths = jax.vmap(make_interp)(test_data)

    all_preds = []
    all_trues = []
    total_loss = 0.0

    num_batches = len(test_data) // batch_size + (1 if len(test_data) % batch_size != 0 else 0)

    for i in range(num_batches):
        start_idx = i * batch_size
        end_idx = min((i + 1) * batch_size, len(test_data))

        batch = test_data[start_idx:end_idx]
        batch_keys = keys[start_idx:end_idx]
        batch_paths = jax.tree.map(lambda x: x[start_idx:end_idx], all_control_paths)

        preds = jax.vmap(
            lambda data, path, k: model(times, path, solver, key=k)[:, 1]
        )(batch, batch_paths, batch_keys)

        trues = batch[:, :, 1]

        all_preds.append(preds)
        all_trues.append(trues)
        total_loss += mse_loss(preds, trues) * len(batch)

    all_preds = jnp.concatenate(all_preds, axis=0)
    all_trues = jnp.concatenate(all_trues, axis=0)
    avg_loss = total_loss / len(test_data)

    return all_preds, all_trues, avg_loss

In [ ]:
# ============================================================================
# INITIALIZATION AND PLOTTING
# ============================================================================
if __name__ == "__main__":
    config = {
        'num_samples': 50000,
        'T': 10.0,
        'N': 20,    # More points = smoother trajectories = MORE RUNNNING TIME

        # OU parameters
        'theta': 0.2,
        'mu': 0.1,
        'sigma': 2.0,

        'X0': 1.0,
        'seed': 42,
        'train_ratio': 0.8,
        'batch_size': 10000,  # SMALLER THE BATCH LONGER IT RUNS. With how we split the data
        'num_epochs': 250,    # (0.8 training, 0.2 validation) this batch_size value means
        'lr': 1e-3,           # there is only one batch iteration

        # Model architecture. More depth = more accuracy = more training time
        # We tried going lower in depth but the model would
        # need more time to learn the dynamics.
        'data_size': 2,
        'noise_size': 32,
        'hidden_size': 32,
        'width_size': 32,
        'depth': 1,
    }

    SOLVERS = [
        ('Euler', diffrax.Euler),
        ('Heun', diffrax.Heun),
        ('Midpoint', diffrax.Midpoint),
        ('EulerHeun', diffrax.EulerHeun),
        ('ReversibleHeun', diffrax.ReversibleHeun),
        ('ItoMilstein', diffrax.ItoMilstein),
        ('StratonovichMilstein', diffrax.StratonovichMilstein),
        ('EES25', diffrax.EES25)
    ]

    BASE_DIR = "plots/memory_multi_solver_comparison"
    os.makedirs(BASE_DIR, exist_ok=True)

    seed_everything(config['seed'])
    base_key = jr.PRNGKey(config['seed'])

    print("=" * 60)
    print("GENERATING OU PROCESS DATA (ONCE)")
    print("=" * 60)

    data_key = jr.fold_in(base_key, 1)
    total_data, times = generate_data(config, data_key)

    split_key = jr.fold_in(base_key, 2)
    train_data, test_data = split_data(total_data, config['train_ratio'], split_key)

    print(f"Train data shape: {train_data.shape}")
    print(f"Test data shape: {test_data.shape}")
    print(f"Times range: [{times[0]}, {times[-1]}]")

    plt.plot(times, total_data[0, :, 1])
    plt.xlabel('Time')
    plt.ylabel('Value')
    plt.title('OU Process Sample Path')
    plt.grid(True)
    plt.savefig(BASE_DIR + "/sample_path.png")
    plt.close()

    print("Plotting total sample paths...")
    cmap = plt.cm.autumn  # or 'YlOrRd', 'Oranges'
    for i, data in enumerate(total_data):
        plt.plot(times, data[:, 1], alpha=0.04, color=cmap(i / len(total_data)))
    plt.plot(times, total_data[0, :, 1], color='darkviolet', linewidth=2, label='Reference path')
    plt.xlabel('Time')
    plt.ylabel('Value')
    plt.title('OU Process Total Sample Paths')
    plt.legend()
    plt.grid(True)
    plt.savefig(BASE_DIR + "/total_sample_path.png")
    plt.close()

    all_results = {}

    for solver_name, solver_cls in SOLVERS:
        print("\n" + "=" * 60)
        print(f"TRAINING WITH SOLVER: {solver_name}")
        print("=" * 60)

        solver_dir = os.path.join(BASE_DIR, solver_name)
        os.makedirs(solver_dir, exist_ok=True)

        try:
            solver = solver_cls()
            test_key = jr.fold_in(base_key, hash(solver_name) % (2**32))
            test_model = NeuralSDE(
                data_size=config['data_size'],
                noise_size=config['noise_size'],
                hidden_size=config['hidden_size'],
                width_size=config['width_size'],
                depth=config['depth'],
                key=test_key
            )
            first_sample = train_data[0]
            t_row = first_sample[:, 0]
            x_row = first_sample[:, 0:2]
            coeffs = diffrax.backward_hermite_coefficients(t_row, x_row)
            control_path = diffrax.CubicInterpolation(t_row, coeffs)
            _ = test_model(times, control_path, solver, key=jr.fold_in(test_key, 999))
        except Exception as e:
            print(f"❌ Solver {solver_name} failed compatibility test: {e}")
            all_results[solver_name] = {"error": str(e)}
            continue

        print(f"✓ Solver {solver_name} is compatible")

        model_key = jr.fold_in(base_key, 100 + hash(solver_name) % (2**32))
        model = NeuralSDE(
            data_size=config['data_size'],
            noise_size=config['noise_size'],
            hidden_size=config['hidden_size'],
            width_size=config['width_size'],
            depth=config['depth'],
            key=model_key
        )

        optimizer = optax.adam(config['lr'])
        opt_state = optimizer.init(eqx.filter(model, eqx.is_array))

        eval_key = jr.fold_in(model_key, 1)
        all_preds, all_trues, test_loss = evaluate_model(model, test_data[:1000], times, eval_key, solver)
        print(f'Initial Test Loss: {test_loss:.6f}')

        print("Measuring compilation time and memory...")
        warmup_batch = train_data[:config['batch_size']]
        warmup_key = jr.fold_in(model_key, 0)

        tracemalloc.start()
        gpu_compile_start = get_gpu_memory_mb()

        # Start background monitoring for compilation
        compile_monitor = BackgroundMemoryMonitor(interval=0.01)
        compile_monitor.start_monitoring()

        compile_start = time.perf_counter()
        model, opt_state, warmup_loss = make_step(
            model, opt_state, optimizer, warmup_batch, times, warmup_key, solver
        )
        jax.block_until_ready((model, opt_state, warmup_loss))
        compile_time = time.perf_counter() - compile_start

        compile_cpu_current, compile_cpu_peak = tracemalloc.get_traced_memory()
        gpu_compile_end = get_gpu_memory_mb()
        compile_gpu_peak = compile_monitor.stop_monitoring()

        compile_mem = {
            'cpu_peak_mb': compile_cpu_peak / (1024 * 1024),
            'cpu_peak_bytes': compile_cpu_peak,
            'gpu_mb': gpu_compile_end,
            'gpu_peak_mb': compile_gpu_peak,
            'gpu_used_mb': gpu_compile_end - gpu_compile_start
        }

        print(f"Compilation time: {compile_time:.2f}s")
        print(f"Compilation CPU memory: {format_memory(compile_mem['cpu_peak_bytes'])}")
        print(f"Compilation GPU memory (end): {compile_mem['gpu_mb']:.2f} MB")
        print(f"Compilation GPU memory (peak): {compile_mem['gpu_peak_mb']:.2f} MB")

        mse_losses = []
        train_start = time.perf_counter()
        gpu_train_start = gpu_compile_end

        # Start background monitoring for training
        gpu_monitor = BackgroundMemoryMonitor(interval=0.01)
        gpu_monitor.start_monitoring()

        for epoch in range(1, config['num_epochs'] + 1):
            epoch_key = jr.fold_in(model_key, epoch)

            perm_key, batch_key = jr.split(epoch_key)
            perm = jr.permutation(perm_key, len(train_data))
            train_data_shuffled = train_data[perm]

            data_loader = create_data_loader(
                train_data_shuffled,
                config['batch_size'],
                shuffle=False,
                key=None
            )

            epoch_loss = 0.0
            num_batches = 0

            for batch in data_loader:
                step_key = jr.fold_in(batch_key, num_batches)
                model, opt_state, loss = make_step(
                    model, opt_state, optimizer, batch, times, step_key, solver
                )
                jax.block_until_ready((model, opt_state, loss))

                # Record step for time-series (background thread captures peaks)
                gpu_monitor.record_step()

                epoch_loss += loss
                num_batches += 1

            if num_batches > 0:
                avg_loss = epoch_loss / num_batches
                mse_losses.append(float(avg_loss))

            if epoch % 10 == 0:
                elapsed_so_far = time.perf_counter() - train_start

                print(f'Epoch {epoch}, Loss: {avg_loss:.6f}, Current GPU: {get_gpu_memory_mb():.1f} MB, Peak so far: {gpu_monitor.peak:.1f} MB')

                eval_key = jr.fold_in(model_key, epoch + 1000)
                all_preds, all_trues, test_loss = evaluate_model(model, test_data[:1000], times, eval_key, solver)
                print(f'Test Loss: {test_loss:.6f}')

                num_samples = 5
                plt.figure(figsize=(8, 4))
                for i in range(num_samples):
                    plt.plot(times, all_trues[i], color='r', alpha=0.7, label='True' if i==0 else None)
                    plt.plot(times, all_preds[i], color='b', alpha=0.7, label='Pred' if i==0 else None)
                plt.xlabel('Time')
                plt.ylabel('Value')
                plt.ylim(-0.75, 1.25)
                plt.title(f'{solver_name} - Epoch {epoch}')
                plt.legend()
                plt.savefig(solver_dir + f"/model_pred{epoch}.png")
                plt.close()

                train_start = time.perf_counter() - elapsed_so_far

        jax.block_until_ready((model, opt_state))
        train_time = time.perf_counter() - train_start

        train_cpu_current, train_cpu_peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        gpu_train_end = get_gpu_memory_mb()

        # Stop background monitoring and get true peak
        gpu_training_peak = gpu_monitor.stop_monitoring()
        gpu_peak_delta = gpu_training_peak - gpu_train_start
        gpu_steps, gpu_history = gpu_monitor.get_history()

        train_mem = {
            'cpu_peak_mb': train_cpu_peak / (1024 * 1024),
            'cpu_peak_bytes': train_cpu_peak,
            'gpu_mb': gpu_train_end,
            'gpu_peak_mb': gpu_training_peak,
            'gpu_peak_delta_mb': gpu_peak_delta,
            'gpu_history': (gpu_steps, gpu_history),
            'gpu_used_mb': gpu_train_end - gpu_train_start
        }

        train_cpu_added = max(0, train_cpu_peak - compile_cpu_peak)
        train_cpu_added_mb = train_cpu_added / (1024 * 1024)
        train_gpu_added_mb = max(0, gpu_train_end - gpu_compile_end)

        first_run_time = compile_time + train_time

        import sys

        print(f"\n{'='*60}", flush=True)
        print(f"RESULTS - {solver_name}", flush=True)
        print(f"{'='*60}", flush=True)
        print(f"Compilation time:          {compile_time:.2f}s", flush=True)
        print(f"Compilation CPU memory:    {format_memory(compile_mem['cpu_peak_bytes'])}", flush=True)
        print(f"Compilation GPU peak:      {compile_mem['gpu_peak_mb']:.2f} MB", flush=True)
        print(f"Training time:             {train_time:.2f}s", flush=True)
        print(f"Total CPU peak memory:     {format_memory(train_mem['cpu_peak_bytes'])}", flush=True)
        print(f"Total GPU memory (end):    {train_mem['gpu_mb']:.2f} MB", flush=True)
        print(f"Peak GPU during training:  {train_mem['gpu_peak_mb']:.2f} MB ← TRUE PEAK", flush=True)
        print(f"GPU peak above baseline:   {train_mem['gpu_peak_delta_mb']:.2f} MB", flush=True)
        print(f"Training added (CPU):      {format_memory(train_cpu_added)}", flush=True)
        print(f"Training added (GPU):      {train_gpu_added_mb:.2f} MB", flush=True)
        print(f"First run total:           {first_run_time:.2f}s", flush=True)
        print(f"{'='*60}\n", flush=True)
        sys.stdout.flush()

        eval_key = jr.fold_in(model_key, 9999)
        all_preds, all_trues, final_test_loss = evaluate_model(model, test_data[:1000], times, eval_key, solver)

        with open(solver_dir + '/mse.pickle', 'wb') as handle:
            pickle.dump(mse_losses, handle)

        plt.plot(mse_losses)
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.title(f'Training Loss - {solver_name}')
        plt.grid(True)
        plt.yscale('log')
        plt.savefig(solver_dir + "/loss.png")
        plt.close()

        all_results[solver_name] = {
            "compile_time": compile_time,
            "compile_cpu_memory_mb": compile_mem['cpu_peak_mb'],
            "compile_cpu_memory_bytes": compile_mem['cpu_peak_bytes'],
            "compile_gpu_memory_mb": compile_mem['gpu_mb'],
            "compile_gpu_peak_mb": compile_mem['gpu_peak_mb'],
            "train_time": train_time,
            "train_cpu_memory_mb": train_mem['cpu_peak_mb'],
            "train_cpu_memory_bytes": train_mem['cpu_peak_bytes'],
            "train_gpu_memory_mb": train_mem['gpu_mb'],
            "train_gpu_peak_mb": train_mem['gpu_peak_mb'],
            "train_gpu_peak_delta_mb": train_mem['gpu_peak_delta_mb'],
            "train_cpu_added_mb": train_cpu_added_mb,
            "train_gpu_added_mb": train_gpu_added_mb,
            "gpu_memory_history": (gpu_steps, gpu_history),
            "first_run_time": first_run_time,
            "final_test_loss": float(final_test_loss),
            "losses": mse_losses
        }

    successful_solvers = {k: v for k, v in all_results.items() if "error" not in v}

    if len(successful_solvers) > 0:
        print("\n" + "=" * 60)
        print("CREATING COMPARISON PLOTS")
        print("=" * 60)

        names = list(successful_solvers.keys())

        # Loss curves
        plt.figure(figsize=(10, 6))
        for name in names:
            plt.plot(successful_solvers[name]["losses"], label=name)
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.yscale('log')
        plt.title('Training Loss Comparison')
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(BASE_DIR + "/loss_comparison.png")
        plt.close()

        # Time comparisons
        train_times = [successful_solvers[n]["train_time"] for n in names]
        compile_times = [successful_solvers[n]["compile_time"] for n in names]
        first_run_times = [successful_solvers[n]["first_run_time"] for n in names]

        plt.figure(figsize=(10, 5))
        plt.bar(names, train_times, color='steelblue')
        plt.xlabel('Solver')
        plt.ylabel('Training Time (s)')
        plt.title('Training Time Comparison')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.savefig(BASE_DIR + "/train_time_comparison.png")
        plt.close()

        plt.figure(figsize=(10, 5))
        plt.bar(names, compile_times, color='coral')
        plt.xlabel('Solver')
        plt.ylabel('Compilation Time (s)')
        plt.title('Compilation Time Comparison')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.savefig(BASE_DIR + "/compile_time_comparison.png")
        plt.close()

        plt.figure(figsize=(10, 5))
        plt.bar(names, first_run_times, color='mediumseagreen')
        plt.xlabel('Solver')
        plt.ylabel('Total Time (s)')
        plt.title('First Run Total Time')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.savefig(BASE_DIR + "/first_run_time_comparison.png")
        plt.close()

        plt.figure(figsize=(10, 5))
        plt.bar(names, compile_times, label='Compilation', color='coral')
        plt.bar(names, train_times, bottom=compile_times, label='Training', color='steelblue')
        plt.xlabel('Solver')
        plt.ylabel('Time (s)')
        plt.title('Time Breakdown: Compilation vs Training')
        plt.legend()
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.savefig(BASE_DIR + "/time_breakdown.png")
        plt.close()

        # GPU Memory comparisons - using TRUE PEAKS
        train_gpu_peak_mb = [successful_solvers[n]["train_gpu_peak_mb"] for n in names]
        compile_gpu_peak_mb = [successful_solvers[n]["compile_gpu_peak_mb"] for n in names]
        train_gpu_peak_delta_mb = [successful_solvers[n]["train_gpu_peak_delta_mb"] for n in names]

        plt.figure(figsize=(10, 5))
        plt.bar(names, train_gpu_peak_mb, color='purple')
        plt.xlabel('Solver')
        plt.ylabel('GPU Memory (MB)')
        plt.title('Peak GPU Memory Usage (Background Monitored)')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.savefig(BASE_DIR + "/gpu_memory_comparison.png")
        plt.close()

        # CPU Memory comparisons
        train_cpu_memory_mb = [successful_solvers[n]["train_cpu_memory_mb"] for n in names]

        plt.figure(figsize=(10, 5))
        plt.bar(names, train_cpu_memory_mb, color='orange')
        plt.xlabel('Solver')
        plt.ylabel('CPU Memory (MB)')
        plt.title('Total CPU Memory Usage')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.savefig(BASE_DIR + "/cpu_memory_comparison.png")
        plt.close()

        # GPU Memory breakdown
        plt.figure(figsize=(10, 5))
        plt.bar(names, compile_gpu_peak_mb, label='Compilation peak', color='coral')
        plt.bar(names, train_gpu_peak_delta_mb, bottom=compile_gpu_peak_mb,
                label='Training peak delta', color='purple')
        plt.xlabel('Solver')
        plt.ylabel('GPU Memory (MB)')
        plt.title('GPU Memory Breakdown: Compilation vs Training Peak')
        plt.legend()
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.savefig(BASE_DIR + "/gpu_memory_breakdown.png")
        plt.close()

        # GPU Memory Over Time Line Graph
        plt.figure(figsize=(14, 6))
        for name in names:
            steps, memory = successful_solvers[name]["gpu_memory_history"]
            if len(steps) > 0:
                plt.plot(steps, memory, label=name, alpha=0.8, linewidth=2)

        plt.xlabel('Training Step (Batch Number)', fontsize=12)
        plt.ylabel('GPU Memory (MB)', fontsize=12)
        plt.title('GPU Memory Usage Over Time - All Solvers (Sampled after each batch)', fontsize=14, fontweight='bold')
        plt.legend(loc='best', fontsize=10)
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(BASE_DIR + "/gpu_memory_over_time.png", dpi=150)
        plt.close()

        # Individual solver memory traces
        fig, axes = plt.subplots(2, 4, figsize=(16, 8))
        axes = axes.flatten()

        for idx, name in enumerate(names):
            if idx < len(axes):
                steps, memory = successful_solvers[name]["gpu_memory_history"]
                if len(steps) > 0:
                    axes[idx].plot(steps, memory, color='purple', linewidth=1.5)
                    axes[idx].axhline(y=successful_solvers[name]["train_gpu_peak_mb"],
                                     color='red', linestyle='--', linewidth=1,
                                     label=f'Peak: {successful_solvers[name]["train_gpu_peak_mb"]:.1f} MB')
                    axes[idx].set_title(name, fontweight='bold')
                    axes[idx].set_xlabel('Step')
                    axes[idx].set_ylabel('GPU Memory (MB)')
                    axes[idx].grid(True, alpha=0.3)
                    axes[idx].legend(fontsize=8)

        # Hide unused subplots
        for idx in range(len(names), len(axes)):
            axes[idx].axis('off')

        plt.tight_layout()
        plt.savefig(BASE_DIR + "/gpu_memory_individual_traces.png", dpi=150)
        plt.close()

        # Time vs GPU Memory scatter plots
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

        ax1.scatter(train_times, train_gpu_peak_mb, s=100, alpha=0.6,
                   c=range(len(names)), cmap='viridis')
        for i, name in enumerate(names):
            ax1.annotate(name, (train_times[i], train_gpu_peak_mb[i]),
                        fontsize=8, ha='right', va='bottom')
        ax1.set_xlabel('Training Time (s)')
        ax1.set_ylabel('Peak GPU Memory (MB)')
        ax1.set_title('Training: Time vs Peak GPU Memory Tradeoff')
        ax1.grid(True, alpha=0.3)

        ax2.scatter(compile_times, compile_gpu_peak_mb, s=100, alpha=0.6,
                   c=range(len(names)), cmap='viridis')
        for i, name in enumerate(names):
            ax2.annotate(name, (compile_times[i], compile_gpu_peak_mb[i]),
                        fontsize=8, ha='right', va='bottom')
        ax2.set_xlabel('Compilation Time (s)')
        ax2.set_ylabel('Peak GPU Memory (MB)')
        ax2.set_title('Compilation: Time vs Peak GPU Memory Tradeoff')
        ax2.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig(BASE_DIR + "/time_gpu_memory_tradeoff.png")
        plt.close()

        # Final test loss comparison
        final_losses = [successful_solvers[n]["final_test_loss"] for n in names]
        plt.figure(figsize=(10, 5))
        plt.bar(names, final_losses)
        plt.xlabel('Solver')
        plt.ylabel('Final Test Loss')
        plt.title('Final Test Loss Comparison')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.savefig(BASE_DIR + "/final_loss_comparison.png")
        plt.close()

        # Print summary
        print("\n" + "=" * 125)
        print("SUMMARY - TRUE PEAK GPU MEMORY (Background Monitored @ 100 Hz)")
        print("=" * 125)
        print(f"{'Solver':<20} | {'Compile':<8} | {'Train':<8} | {'GPU-C-Peak':<11} | {'GPU-T-Peak':<11} | {'GPU-Delta':<10} | {'Loss':<12}")
        print(f"{'':20} | {'Time':<8} | {'Time':<8} | {'(MB)':<11} | {'(MB)':<11} | {'(MB)':<10} | {'':12}")
        print("-" * 125)
        for name in names:
            print(f"{name:<20} | "
                  f"{successful_solvers[name]['compile_time']:>6.2f}s | "
                  f"{successful_solvers[name]['train_time']:>6.2f}s | "
                  f"{successful_solvers[name]['compile_gpu_peak_mb']:>9.2f} | "
                  f"{successful_solvers[name]['train_gpu_peak_mb']:>9.2f} | "
                  f"{successful_solvers[name]['train_gpu_peak_delta_mb']:>8.2f} | "
                  f"{successful_solvers[name]['final_test_loss']:.8f}")

        print("\n" + "=" * 125)
        print("INTERPRETATION GUIDE")
        print("=" * 125)
        print("Compile Time:    One-time cost when first running the solver")
        print("Train Time:      Recurring cost for each training run (what matters most!)")
        print("GPU-C-Peak:      TRUE peak GPU memory during compilation (background monitored)")
        print("GPU-T-Peak:      TRUE peak GPU memory during training (background monitored)")
        print("GPU-Delta:       Additional peak memory during training (T-Peak - C-Peak)")
        print("Loss:            Model accuracy on test set")
        if len(names) > 0:
            print(f"\nNote: CPU memory ~{train_cpu_memory_mb[0]:.1f} MB (tracked separately)")
        print("\nNEW: Background monitoring at 100 Hz captures TRUE peaks during computation!")
        print("     Check 'gpu_memory_over_time.png' to see memory usage during training!")
        print("     Check 'gpu_memory_individual_traces.png' for per-solver details!")
        print("\nFor repeated experiments → focus on 'Train Time' and 'GPU-Delta'")
        print("For memory-constrained GPUs → focus on 'GPU-T-Peak'")
        print("=" * 125)

    # Save all results
    with open(BASE_DIR + '/all_results.pickle', 'wb') as handle:
        pickle.dump(all_results, handle)

    # Cleanup GPU if available
    if GPU_AVAILABLE:
        try:
            pynvml.nvmlShutdown()
        except:
            pass

    print(f"\n✓ All results saved to {BASE_DIR}/")
    print("Done!")